# NB1 - VL04 Batteriemodellierung und Batteriesystemtechnik

Dieses Notebook behandelt zentrale Grundlagen der Batteriemodellierung auf Zellebene und die Übertragung auf eine einfache Systemtopologie.

Schwerpunkte des Notebooks:
- Beispielhafte Leistungsprofile
- Reservoirmodell
- Ersatzschaltbildmodell (ECM)
- Kennzahlen und C-Rate
- Einfache Topologie von der Zelle zum System


In [1]:
# Import necessary libraries. Run this cell once before executing the notebook.
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy.optimize import fsolve
from plotly.subplots import make_subplots


In [2]:
# Configure Plotly defaults. Run this cell once before working with the notebook.
pd.options.plotting.backend = "plotly"
template = "plotly_white"


## 1. Beispielhafte Leistungsprofile

Ein Leistungsprofil beschreibt, wie sich die Leistung über der Zeit verändert, also welches Lade- und Entlademuster auf ein System wirkt.
In diesem Notebook dient es als gemeinsamer Eingang für das Reservoirmodell und das Ersatzschaltbildmodell, damit beide Ansätze unter identischen Bedingungen untersucht werden können.
So lassen sich Reaktionsverhalten, Grenzen und Dynamik der Modelle direkt vergleichen.


### 1.1. Erzeugungsfunktion

Die folgende Funktion erzeugt ein Leistungsprofil, das in festen Zeitabständen zwischen positiven und negativen Leistungswerten wechselt.


In [3]:
def generate_power_profile1(duration_d, time_step_d, power_level_w, switch_interval_d):
    # The following string is stored as the function's docstring and
    # can be accessed via help(generate_power_profile1) or generate_power_profile1.__doc__
    """
    Generates a stationary power profile that periodically switches
    between +power_level_w and -power_level_w.

    Parameters
    ----------
    duration_d : int or float
        Total simulation time in days.
    time_step_d : float
        Time step in days.
    power_level_w : float
        Maximum positive power level (battery takes energy, i.e. charge) and
        maximum negative power level (battery gives energy, i.e. discharge) in Watts.
    switch_interval_d : float
        Interval in days after which the power sign switches.
        If 0, the value never switches.

    Returns
    -------
    power_profile_time_d : np.ndarray
        Time vector [days].
    power_profile_w : np.ndarray
        Power profile [W], corresponding to `power_profile_time_d`.
    """
    if switch_interval_d == 0:
        switch_interval_d = duration_d

    power_profile_time_d = np.arange(0, duration_d, time_step_d)
    sign = np.where((power_profile_time_d // switch_interval_d) % 2 == 0, 1, -1)
    power_profile_w = sign * power_level_w

    return power_profile_time_d, power_profile_w

### 1.3. Visualisierung des Leistungsprofils

Die folgende Funktion stellt das erzeugte Leistungsprofil mit Plotly dar.


In [4]:
def plot_power_profile(power_profile_time_d, power_profile_w, title="Example power profile"):
    """
    Plots a generated power profile.

    Parameters
    ----------
    power_profile_time_d : np.ndarray
        Time vector [days].
    power_profile_w : np.ndarray
        Power profile [W].
    title : str, optional
        Title of the plot.

    Examples
    --------
    >>> power_profile_time_d, power_profile_w = generate_power_profile1(
    ...     duration_d=60,
    ...     time_step_d=1,
    ...     power_level_w=100,
    ...     switch_interval_d=10,
    ... )
    >>> plot_power_profile(power_profile_time_d, power_profile_w, title="Periodisches Lade-/Entladeprofil")
    """
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=power_profile_time_d,
            y=power_profile_w,
            mode="lines",
            name="Power [W]",
            line=dict(width=2, shape="hv"),
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Time [days]",
        yaxis_title="Power [W]",
        template="plotly_white",
        hovermode="x unified",
    )
    fig.show()

Nun kann das erzeugte Leistungsprofil visualisiert werden. Dadurch wird sichtbar, wie sich die Last bzw. Ladeanforderung zeitlich ändert.

In [5]:
# Define power_profile_time_d_1 and power_profile_w_1 with generate_power_profile1()
# and call plot_power_profile() with these variables
power_profile_time_d_1, power_profile_w_1 = generate_power_profile1(
    duration_d=7,
    time_step_d=0.1,
    power_level_w=50,
    switch_interval_d=1,
)
plot_power_profile(power_profile_time_d_1, power_profile_w_1)

## 2. Batteriemodellierung: Reservoirmodell und ECM

### 2.1. Reservoirmodell

Das **Reservoirmodell** (Res-M, auch **Bucket-Modell**) ist ein einfacher Ansatz zur Beschreibung eines Energiespeichers. Der Speicher wird dabei wie ein Behälter betrachtet, der mit Energie „gefüllt“ und „entleert“ werden kann. Für Batterien bedeutet das:

- **Laden** erhöht den Füllstand des Speichers.
- **Entladen** verringert den Füllstand des Speichers.
- Der **Füllstand** entspricht dem **State of Energy (SoE)**.
- Die **Behältergröße** entspricht der **Energiekapazität**.

Das Modell eignet sich gut, um Energiebilanz, Grenzen und Verfügbarkeit anschaulich zu erklären, bevor komplexere Modelle wie das ECM eingeführt werden.


#### 2.1.1. Zellparameter des Reservoirmodells

In [6]:
example_system_energy_capacity_wh = 444  # Wh
example_system_round_trip_efficiency_pu = 0.95  # p.u.
example_system_max_power_w = 480  # W
example_system_initial_soe_pu = 0.5  # p.u.

#### 2.1.2. Clamp-Funktion

Die Clamp-Funktion stellt sicher, dass

- der **State of Energy (SoE)** stets im physikalisch sinnvollen Bereich zwischen 0 und 1 bleibt und
- die **reale Leistung** innerhalb der technischen Grenzen des Systems liegt, also zwischen `-max_power` und `+max_power`.

Dadurch verletzt das Modell während der Simulation weder physikalische noch technische Randbedingungen.


In [7]:
def clamp(lower_bound, value, upper_bound):
    """
    Restricts a numeric value to lie within given lower and upper limits.

    Parameters
    ----------
    lower_bound : float or int
        Minimum allowed value. If `value` is smaller than this limit,
        the function returns `lower_bound`.
    value : float or int
        Input value to be clamped.
    upper_bound : float or int
        Maximum allowed value. If `value` exceeds this limit,
        the function returns `upper_bound`.

    Returns
    -------
    float or int
        The clamped value, constrained between `lower_bound` and `upper_bound`.

    Examples
    --------
    >>> clamp(0, -3, 10)
    0
    >>> clamp(0, 5, 10)
    5
    >>> clamp(0, 15, 10)
    10
    """
    return min(max(value, lower_bound), upper_bound)

#### 2.1.3. Update-Funktion

`update_reservoir_state()` bildet den physikalischen Kern des Reservoirmodells für einen einzelnen Zeitschritt. Die Funktion beschreibt also, was innerhalb eines Simulationsschritts passiert: Leistungsbegrenzung, SoE-Aktualisierung, Ausnutzungsgrad und weitere Kenngrößen.

Die Kapselung in einer eigenen Funktion erleichtert Tests, Erweiterungen und die spätere Auswertung von Kennzahlen wie Wirkungsgrad, Verfügbarkeit oder Verlusten.



<div class="alert alert-block alert-success">

**Hinweis zur Vorzeichenkonvention**

In diesem Notebook gilt durchgängig:

- `power_target > 0`  → Laden
- `power_target < 0`  → Entladen

Dieselbe Konvention wird im Reservoirmodell und im ECM verwendet.

</div>


In [8]:
def update_reservoir_state(soe_pu, power_target_w, system_energy_capacity_wh, system_max_power_w, time_step_h):
    """
    Update the state of energy (SoE) of a storage unit with a simple reservoir model.

    Parameters
    ----------
    soe_pu : float
        Current state of energy, normalized between 0 (empty) and 1 (full).
    power_target_w : float
        Requested power in W. Positive values charge the battery,
        negative values discharge the battery.
    system_energy_capacity_wh : float
        Total energy capacity of the storage in Wh.
    system_max_power_w : float
        Maximum absolute charging or discharging power in W.
    time_step_h : float
        Time step in h.

    Returns
    -------
    dict
        Dictionary with updated SoE, actual power, and fulfillment ratio.
    """
    power_w = clamp(-system_max_power_w, power_target_w, system_max_power_w)
    soe_pu_new = clamp(0, soe_pu + power_w * time_step_h / system_energy_capacity_wh, 1)
    power_w = round((soe_pu_new - soe_pu) / time_step_h * system_energy_capacity_wh, 4)

    if power_target_w == 0:
        power_fulfillment_pu = np.nan
    else:
        power_fulfillment_pu = round(power_w / power_target_w, 4)

    return {
        "soe_pu": soe_pu_new,
        "power_w": power_w,
        "power_fulfillment_pu": power_fulfillment_pu,
    }

#### 2.1.4. Simulationsfunktion

`simulate_reservoir()` ist der zeitliche Rahmen um das Reservoirmodell: Die Funktion führt `update_reservoir_state()` über viele Zeitschritte aus, verwaltet die Schleife und sammelt die Ergebnisse in einem DataFrame.

**Wichtig:** `energy_capacity` wird hier in **Wh** verwendet. Eine Verwechslung von Zellkapazität in **Ah** mit Energie in **Wh** führt unmittelbar zu falschen Ergebnissen.

**Hinweis**  Die Iteration erfolgt direkt über `power_profile.items()`, damit Zeitindex und Zielwert sauber gemeinsam verarbeitet werden.

</div>

In [9]:
def simulate_reservoir(power_profile_w, system_energy_capacity_wh, system_max_power_w, initial_soe_pu, time_step_h):
    """
    Simulate the reservoir model over a complete power profile.

    Parameters
    ----------
    power_profile_w : pandas.Series
        Power profile with time as index and power values in W.
    system_energy_capacity_wh : float
        Total energy capacity of the storage in Wh.
    system_max_power_w : float
        Maximum absolute charging or discharging power in W.
    initial_soe_pu : float
        Initial state of energy, normalized between 0 and 1.
    time_step_h : float
        Time step in h.

    Returns
    -------
    pandas.DataFrame
        Simulation results containing target power, applied power, SoE, and fulfillment.
    """
    power_profile_w = pd.to_numeric(power_profile_w, errors="coerce")

    df_result = pd.DataFrame(index=power_profile_w.index)
    df_result["power_target_w"] = power_profile_w.astype(float)
    df_result["power_w"] = np.nan
    df_result["soe_pu"] = np.nan
    df_result["power_fulfillment_pu"] = np.nan

    soe_pu = float(initial_soe_pu)

    for time_value, power_target_w in power_profile_w.items():
        if pd.isna(power_target_w):
            df_result.loc[time_value, "power_w"] = np.nan
            df_result.loc[time_value, "soe_pu"] = soe_pu
            df_result.loc[time_value, "power_fulfillment_pu"] = np.nan
            continue

        state_update = update_reservoir_state(
            soe_pu=soe_pu,
            power_target_w=float(power_target_w),
            system_energy_capacity_wh=system_energy_capacity_wh,
            system_max_power_w=system_max_power_w,
            time_step_h=time_step_h,
        )

        power_w = float(state_update["power_w"]) if state_update["power_w"] is not None else np.nan
        soe_pu_new = float(state_update["soe_pu"]) if state_update["soe_pu"] is not None else np.nan
        power_fulfillment_pu = (
            float(state_update["power_fulfillment_pu"])
            if state_update["power_fulfillment_pu"] is not None
            else np.nan
        )

        df_result.loc[time_value, "power_w"] = power_w
        df_result.loc[time_value, "soe_pu"] = soe_pu_new
        df_result.loc[time_value, "power_fulfillment_pu"] = power_fulfillment_pu

        if not pd.isna(soe_pu_new):
            soe_pu = soe_pu_new

    return df_result

#### 2.1.5. Visualisierung

Die folgenden Funktionen visualisieren die Ergebnisse des Reservoirmodells mit Plotly. Dargestellt werden insbesondere Leistung, Leistungsvorgabe, SoE und Erfüllungsgrad.


In [10]:
def plot_reservoir(df_result, title="Reservoir Model Simulation"):
    """
    Plot reservoir-model results with Plotly.

    Parameters
    ----------
    df_result : pandas.DataFrame
        DataFrame with at least the columns
        ['power_target_w', 'power_w', 'soe_pu', 'power_fulfillment_pu'].
    title : str
        Plot title.
    """
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df_result.index,
            y=df_result["power_w"],
            name="Power [W]",
            mode="lines",
            line=dict(color="firebrick"),
            yaxis="y1",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df_result.index,
            y=df_result["soe_pu"],
            name="State of Energy (SoE) [p.u.]",
            mode="lines",
            line=dict(color="royalblue"),
            yaxis="y2",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df_result.index,
            y=df_result["power_target_w"],
            name="Power Target [W]",
            mode="lines",
            line=dict(color="gray", dash="dot"),
            yaxis="y1",
        )
    )

    fig.update_layout(
        title=title,
        xaxis=dict(title="Time [days]"),
        yaxis=dict(title="Power [W]", side="left", showgrid=False),
        yaxis2=dict(
            title="State of Energy [p.u.]",
            side="right",
            overlaying="y",
            range=[0, 1],
            showgrid=False,
        ),
        template="plotly_white",
        height=500,
    )

    fig.show()

In [11]:
def plot_fulfillment(df_result, title="Fulfillment"):
    """
    Plot the fulfillment ratio over time.

    Parameters
    ----------
    df_result : pandas.DataFrame
        DataFrame containing a 'power_fulfillment_pu' column.
    title : str
        Plot title.
    """
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df_result.index,
            y=df_result["power_fulfillment_pu"],
            name="Fulfillment",
            mode="lines",
            line=dict(color="royalblue", shape="hv"),
        )
    )

    fig.update_layout(
        title=title,
        xaxis=dict(title="Time [days]"),
        yaxis=dict(
            title="Fulfillment [p.u.]",
            range=[0, max(1, np.nanmax(df_result["power_fulfillment_pu"])) if len(df_result) else 1],
            showgrid=False,
        ),
        template="plotly_white",
        height=500,
    )

    fig.show()

In [12]:
power_profile_w = pd.Series(power_profile_w_1, index=power_profile_time_d_1)
time_step_h = (power_profile_time_d_1[1] - power_profile_time_d_1[0]) * 24

Wir fügen nun alle Bestandteile zusammen und führen die Simulation mit dem erzeugten Leistungsprofil aus:

- Simulieren des Batterieverhalten mit dem Reservoirmodell `df_result=simulate_reservoir(...)`.
- Darstellen der Ergebnisse mit `plot_reservoir(...)` und `plot_fulfillment(...)`.


In [13]:
df_result = simulate_reservoir(
    power_profile_w=power_profile_w,
    system_energy_capacity_wh=example_system_energy_capacity_wh,
    system_max_power_w=example_system_max_power_w,
    initial_soe_pu=example_system_initial_soe_pu,
    time_step_h=time_step_h,
)

plot_reservoir(df_result)
plot_fulfillment(df_result)


<div class="alert alert-block alert-info">

**AUFGABE 01**

- Erstellen Sie eine Markdown-Zelle und interpretieren Sie den erzeugten Plot kurz.
- Warum stimmen **Zielleistung** (grau, gestrichelt) und **reale Leistung** (rot, durchgezogen) nicht immer überein?
- Warum weicht der Erfüllungsgrad in einzelnen Zeitpunkten von 1 ab?
- Welche Randbedingungen verhindern, dass die Batterie jederzeit die volle Leistung bereitstellt?
- Welche weiteren Einflussfaktoren könnten den Erfüllungsgrad begrenzen?

</div>


<div class="alert alert-block alert-warning">

### SOLUTION 01

Der Plot zeigt das Verhalten des Bucket-Modells unter dem zuvor erzeugten Leistungsprofil.
Die graue, gestrichelte Kurve repräsentiert die gewünschte Zielleistung, also die Leistung, die das System idealerweise aufnehmen oder abgeben soll. Die rote Kurve zeigt dagegen die tatsächlich umgesetzte Leistung des Batteriemodells.

Solange sich der **State of Energy (SoE)** im zulässigen Bereich befindet und die Leistungsgrenzen nicht verletzt werden, folgt das Modell der Vorgabe sehr gut. Nähern sich jedoch die Energiegrenzen des Speichers den Zuständen **leer** oder **voll**, kann die Batterie nicht mehr jede gewünschte Leistung bereitstellen. Dann weicht die reale Leistung von der Sollvorgabe ab und der Erfüllungsgrad sinkt.

Die Abweichungen werden somit vor allem durch **Leistungsgrenzen**, **Kapazitätsgrenzen** und die gewählte **Anfangsbedingung des SoE** bestimmt. 

In erweiterten Modellen kommen zusätzlich Wirkungsgrad, Temperatur, Alterung oder Strombegrenzungen als Einflussgrößen hinzu - im vorleisgenden Fall sind diese jedoch **nicht implementiert**


<div class="alert alert-block alert-info">

**AUFGABE 02**

Erstelle außerdem eine Funktion `calculate_availability(df_result)`, die die Verfügbarkeit in Prozent berechnet. Notieren Sie das Ergebnis zur direkten Eingabe in Moodle

In [14]:
# Solution 02

def calculate_availability(df_result):
    """Calculate availability in percent based on requested and delivered absolute power."""
    if "power_w" in df_result.columns:
        delivered_power_w = df_result["power_w"]
    elif "cell_power_w" in df_result.columns:
        delivered_power_w = df_result["cell_power_w"]
    else:
        raise KeyError("Expected either 'power_w' or 'cell_power_w' in df_result.")

    requested_power_w = df_result["power_target_w"]
    availability_pct = (np.sum(np.abs(delivered_power_w)) / np.sum(np.abs(requested_power_w))) * 100
    return availability_pct

availability_pct = calculate_availability(df_result)
print("Availability (%):", availability_pct)

Availability (%): 34.35714285714286


### 2.2. Ersatzschaltbildmodell (ECM)

#### 2.2.1. Zellparameter von `LIB_type1`

Für eine generische Lithium-Ionen-Zelle `LIB_type1` liegen die folgenden Nennparameter aus einem Datenblatt vor.


In [15]:
# LIB1 nominal battery parameters
lib1_cell_nominal_capacity_ah_ah = 120  # Ah
lib1_cell_nominal_voltage_v_v = 3.7  # V
lib1_cell_max_power_w_w = 900  # W
lib1_cell_nominal_energy_wh = lib1_cell_nominal_capacity_ah_ah * lib1_cell_nominal_voltage_v_v  # Wh

<div class="alert alert-block alert-info">

**AUFGABE 03**

- Berechne aus den gegebenen Spezifikationen den Innenwiderstand nach der **Maximum Power Transfer Condition (MPTC)**.
- Speichern Sie das Ergebnis in `lib1_cell_resistance_mptc_ohm` und geben Sie den Wert mit korrekter Einheit aus.

</div>

In [16]:
# Solution 03 - derived from the maximum power transfer condition (MPTC)

lib1_cell_resistance_mptc_ohm = lib1_cell_nominal_voltage_v_v**2 / lib1_cell_max_power_w_w/4
print("Internal resistance from MPTC per cell (Ohm):", lib1_cell_resistance_mptc_ohm)

Internal resistance from MPTC per cell (Ohm): 0.003802777777777778


#### 2.2.2. Erweiterte Zellparameter von `LIB_type1`

Für dieselbe Batteriezelle stehen zusätzlich die folgenden erweiterten Parameter zur Verfügung:


In [17]:
# Extended parameters for LIB1
lib1_cell_resistance_ohm = 0.00288  # Ohm
lib1_cell_max_charge_c_rate = 1.0  # 1.0 C
lib1_cell_max_discharge_c_rate = 2.0  # 6.6 C
lib1_cell_max_charge_current_a = lib1_cell_nominal_capacity_ah_ah * lib1_cell_max_charge_c_rate
lib1_cell_max_discharge_current_a = lib1_cell_nominal_capacity_ah_ah * lib1_cell_max_discharge_c_rate

<div class="alert alert-block alert-info">

**AUFGABE 04**

- Warum können sich der berechnete Widerstand und der im Datenblatt angegebene Innenwiderstand unterscheiden?
- Diskutiere stichpunktartig, warum `lib1_cell_resistance_mptc_ohm` und `lib1_cell_resistance_ohm` voneinander abweichen.
- Welcher der beiden Werte ist typischerweise größer?

</div>

<div class="alert alert-block alert-warning">

### Solution 04

Der aus der **Maximum Power Transfer Condition (MPTC)** abgeleitete Widerstand ist typischerweise **größer** als der Innenwiderstand, der unter realistischen Betriebsgrenzen aus dem Datenblatt hervorgeht.

Die MPTC beschreibt einen theoretischen Grenzfall mit

- starkem Spannungseinbruch,
- hohen inneren Verlusten und
- einem energetisch ungünstigen Betriebspunkt.

Reale Batterien werden dort nicht betrieben. Stattdessen begrenzen **Stromgrenzen**, **Temperaturgrenzen**, **Spannungsfenster** und **Sicherheitsreserven** den zulässigen Arbeitspunkt deutlich früher.


#### 2.2.3. Leerlaufspannung (OCV) und Interpretation

Aus einem Laborexperiment liegen OCV-Stützstellen vor, die mit einer Polynomial- bzw. Fit-Funktion angenähert wurden:

In [18]:
def lib1_open_circuit_voltage(soc_pu):
    a1 = 3.3479
    a2 = -6.7241
    a3 = 2.5958
    a4 = -61.9684
    b1 = 0.6350
    b2 = 1.4376
    k0 = 4.5868
    k1 = 3.1768
    k2 = -3.8418
    k3 = -4.6932
    k4 = 0.3618
    k5 = 0.9949

    return (
        k0
        + k1 / (1 + np.exp(a1 * (soc_pu - b1)))
        + k2 / (1 + np.exp(a2 * (soc_pu - b2)))
        + k3 / (1 + np.exp(a3 * (soc_pu - 1)))
        + k4 / (1 + np.exp(a4 * soc_pu))
        + k5 * soc_pu
    )

Im nächsten Schritt wird die OCV-Kennlinie der Batteriezelle dargestellt.

In [19]:
soc_grid_pu = np.linspace(start=0.0, stop=1.0, num=1000)
ocv_curve_v_v = lib1_open_circuit_voltage(soc_grid_pu)

fig = px.line(
    x=soc_grid_pu,
    y=ocv_curve_v_v,
    template=template,
    labels={"x": "SOC [p.u.]", "y": "OCV [V]"},
    title="Open Circuit Voltage (OCV) Curve for LIB1"
)

fig.show()


<div class="alert alert-block alert-info">

**AUFGABE 05**

- Welche Lithium-Ionen-Chemie würdest du auf Basis der OCV-Kennlinie vermuten?
- Begründe deine Entscheidung fachlich.
- Ergänze nach Möglichkeit eine geeignete Online-Quelle.

</div>



<div class="alert alert-block alert-warning">

### Solution 05

Die OCV-Kurve von `LIB1` passt qualitativ eher zu einer **NMC-/NCA-ähnlichen** Zelle als zu einer LFP-Zelle, da kein langes, nahezu horizontales Spannungsplateau sichtbar ist. Stattdessen steigt die Spannung über den SoC-Bereich vergleichsweise kontinuierlich an.

</div>


#### 2.2.4. ECM-Update-Modell

<div class="alert alert-block alert-warning">

**AUFGABE 06 --- eher nicht**

Leite mit der in der Vorlesung verwendeten Beziehung zwischen Leistung, OCV, Innenwiderstand und Strom eine Funktion her, die für jeden Zeitschritt den benötigten Strom bestimmt. Du kannst dafür `fsolve` aus `scipy.optimize` verwenden.

Die Funktion soll `simulate_ecm_step(soc_pu, power_target_w, cell_nominal_capacity_ah, cell_max_charge_current_a, cell_max_discharge_current_a, time_step_h, cell_resistance_ohm)` heißen und den aktualisierten Batteriezustand sowie die zugehörigen elektrischen Größen zurückgeben.

** Beachte** 

-  die Stromlimitierung der Zelle soll eingehlaten werden, d.h. `cell_max_charge_current_a` und `cell_max_discharge_current_a`sollen nicht überschritten werden
- Die Spannungsgrenzen der Zelle können in diesem ersten Ansatz vernachlässigt werden - stattdessen soll lediglich sichergestellt werden, dass die Zelle nur Ladezuständie zwischen `SOC=0` und `SOC=1`annimmt.

- Es soll `power_fulfillment_pu` berechnet und im `return` statement für den Nutzer zurückgegeben werden. Dies bildet die prozentuale Erfüllung des angeforderten Leitungsprofils im jeweiligen Zeitschritt ab.
</div>

In [20]:
# Solution - ECM step update function

def simulate_ecm_step(
    soc_pu,
    power_target_w,
    cell_nominal_capacity_ah,
    cell_max_charge_current_a,
    cell_max_discharge_current_a,
    time_step_h,
    cell_resistance_ohm,
    open_circuit_voltage_function=lib1_open_circuit_voltage,
):
    ocv_v = open_circuit_voltage_function(soc_pu)
    current_initial_guess_a = power_target_w / ocv_v if ocv_v != 0 else 0.0

    cell_current_a = fsolve(
        lambda current_value_a: power_target_w - (ocv_v + current_value_a * cell_resistance_ohm) * current_value_a,
        x0=current_initial_guess_a,
    )[0]

    cell_current_a = clamp(
        -cell_max_discharge_current_a,
        cell_current_a,
        cell_max_charge_current_a,
    )

    soc_pu_new = clamp(0, soc_pu + cell_current_a * time_step_h / cell_nominal_capacity_ah, 1)
    cell_current_a = (soc_pu_new - soc_pu) * cell_nominal_capacity_ah / time_step_h  # Recompute the effective current.

    cell_voltage_v = ocv_v + cell_current_a * cell_resistance_ohm
    cell_power_w = cell_voltage_v * cell_current_a

    if power_target_w == 0:
        power_fulfillment_pu = np.nan
    else:
        power_fulfillment_pu = round(cell_power_w / power_target_w, 4)

    return {
        "cell_power_w": cell_power_w,
        "cell_current_a": cell_current_a,
        "cell_voltage_v": cell_voltage_v,
        "ocv_v": ocv_v,
        "soc_pu": soc_pu_new,
        "power_target_w": power_target_w,
        "power_fulfillment_pu": power_fulfillment_pu,
    }

<div class="alert alert-block alert-warning">

### Lösung: Funktion `simulate_ecm_step`

Die Funktion `simulate_ecm_step` berechnet den neuen Batteriezustand auf Basis eines einfachen elektrischen Ersatzschaltbilds mit Leerlaufspannung $U_{\mathrm{ocv}}$ und Innenwiderstand $r$.

Zunächst wird aus dem aktuellen Ladezustand $\mathrm{SOC}$ die Leerlaufspannung bestimmt:

$$
U_{\mathrm{ocv}} = f_{\mathrm{OCV}}(\mathrm{SOC})
$$

Dabei gilt in diesem Notebook die Vorzeichenkonvention:
- $P_{\mathrm{target}} > 0$  $\rightarrow$ Laden
- $P_{\mathrm{target}} < 0$  $\rightarrow$ Entladen
- $I > 0$  $\rightarrow$ Laden
- $I < 0$  $\rightarrow$ Entladen

Danach wird der Strom $I$ so berechnet, dass die angeforderte Leistung $P_{\mathrm{target}}$ erfüllt wird:

$$
P_{\mathrm{target}} = \left(U_{\mathrm{ocv}} + I r\right) I
$$

Anschließend wird der Strom auf die zulässigen Lade- und Entladegrenzen begrenzt:

$$
I = \operatorname{clamp}\left(-I_{\mathrm{DCH,max}},\, I,\, I_{\mathrm{CH,max}}\right)
$$

Mit diesem Strom ergibt sich der neue Ladezustand:

$$
\mathrm{SOC}_{\mathrm{neu}} =
\operatorname{clamp}\left(
0,\,
\mathrm{SOC} + \frac{I\,\Delta t}{C_{\mathrm{nom}}},\,
1
\right)
$$

Danach werden der wirksame Strom, die Klemmenspannung und die tatsächlich umgesetzte Leistung konsistent nachgerechnet.

</div>


Die folgende Funktion nutzt das ECM anschließend über ein gesamtes Leistungsprofil hinweg und führt die Ergebnisse in einem DataFrame zusammen.


In [21]:
def simulate_ecm(
    power_profile_w,
    cell_nominal_capacity_ah,
    cell_max_charge_current_a,
    cell_max_discharge_current_a,
    initial_soc_pu,
    cell_resistance_ohm,
    time_step_h,
    open_circuit_voltage_function=lib1_open_circuit_voltage,
):
    simulation_results = []
    soc_pu = initial_soc_pu

    for time_value, power_target_w in power_profile_w.items():
        step_result = simulate_ecm_step(
            soc_pu=soc_pu,
            power_target_w=power_target_w,
            cell_nominal_capacity_ah=cell_nominal_capacity_ah,
            cell_max_charge_current_a=cell_max_charge_current_a,
            cell_max_discharge_current_a=cell_max_discharge_current_a,
            time_step_h=time_step_h,
            cell_resistance_ohm=cell_resistance_ohm,
            open_circuit_voltage_function=open_circuit_voltage_function,
        )
        simulation_results.append((time_value, step_result))
        soc_pu = step_result["soc_pu"]

    df_result = pd.DataFrame(
        data=[entry[1] for entry in simulation_results],
        index=[entry[0] for entry in simulation_results],
    )

    return df_result

In [22]:
# simulation des ECM modells für das vorgegebene Leistungsprofil und die LIB1 Zellparameter
df_lib1_ecm_result = simulate_ecm(
    power_profile_w=power_profile_w,
    cell_nominal_capacity_ah=lib1_cell_nominal_capacity_ah_ah,
    cell_max_charge_current_a=lib1_cell_max_charge_current_a,
    cell_max_discharge_current_a=lib1_cell_max_discharge_current_a,
    initial_soc_pu=0.5,
    cell_resistance_ohm=lib1_cell_resistance_ohm,
    time_step_h=time_step_h,
)

### 2.2.7. Darstellung Leistungsabruf mit dem ECM Modell

In [23]:
def plot_ecm(df_result, title="ECM Simulation"):
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        specs=[[{"secondary_y": True}], [{"secondary_y": True}]],
    )

    fig.add_trace(
        go.Scatter(
            x=df_result.index,
            y=df_result["soc_pu"],
            mode="lines",
            name="SOC [p.u.]",
            line=dict(width=2),
        ),
        row=1,
        col=1,
        secondary_y=False,
    )

    fig.add_trace(
        go.Scatter(
            x=df_result.index,
            y=df_result["cell_power_w"],
            mode="lines",
            name="Power [W]",
            line=dict(width=2),
        ),
        row=1,
        col=1,
        secondary_y=True,
    )

    fig.add_trace(
        go.Scatter(
            x=df_result.index,
            y=df_result["cell_voltage_v"],
            mode="lines",
            name="Voltage [V]",
            line=dict(width=2),
        ),
        row=2,
        col=1,
        secondary_y=False,
    )

    fig.add_trace(
        go.Scatter(
            x=df_result.index,
            y=df_result["cell_current_a"],
            mode="lines",
            name="Current [A]",
            line=dict(width=2),
        ),
        row=2,
        col=1,
        secondary_y=True,
    )

    fig.update_yaxes(title_text="SOC [p.u.]", range=[0, 1], row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Power [W]", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Voltage [V]", row=2, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Current [A]", row=2, col=1, secondary_y=True)
    fig.update_xaxes(title_text="Time [days]", row=2, col=1)

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=700,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=60, r=60, t=60, b=50),
    )

    fig.show()

In [24]:
plot_ecm(
    # rufe hier die Simulation der LIB1 Zelle auf
    df_lib1_ecm_result,
    title="ECM Simulation for LIB1",
)

#### 2.2.6. Effizienzanalyse und Verfügbarkeit 

In [25]:
def calculate_battery_efficiency(df_result, cell_resistance_ohm):
    cell_current_a = df_result["cell_current_a"]
    cell_power_w = df_result["cell_power_w"]
    discharge_power_w = (-df_result["cell_power_w"]).clip(lower=0)
    thermal_losses_w = np.sum(cell_current_a**2 * cell_resistance_ohm)

    battery_thermal_efficiency_pct = (1 - thermal_losses_w / np.sum(np.abs(cell_power_w))) * 100
    relative_battery_loss_pct = thermal_losses_w * 100 / (np.sum(np.abs(discharge_power_w)) + thermal_losses_w)
    battery_round_trip_efficiency_pct = 100 - relative_battery_loss_pct

    return (
        battery_thermal_efficiency_pct,
        relative_battery_loss_pct,
        battery_round_trip_efficiency_pct,
    )

In [26]:
lib1_battery_thermal_efficiency_pct, lib1_relative_battery_loss_pct, lib1_battery_round_trip_efficiency_pct = calculate_battery_efficiency(
    df_lib1_ecm_result,
    lib1_cell_resistance_ohm,
)
print("Battery Thermal Efficiency (%):", lib1_battery_thermal_efficiency_pct)
print("Battery Round Trip Efficiency (%):", lib1_battery_round_trip_efficiency_pct)

Battery Thermal Efficiency (%): 99.00350933639623
Battery Round Trip Efficiency (%): 97.9250978070516


In [27]:
availability_pct = calculate_availability(df_lib1_ecm_result)
print("Availability LIB1 ECM Model (%):", availability_pct)

Availability LIB1 ECM Model (%): 34.44080798749692



<div class="alert alert-block alert-info">

**AUFGABE 06**

- Analysieren Sie die Werte von  **Battery Thermal Efficiency** und **Battery Round Trip Efficiency**  - Gibt es Abweichungen? Wesshalb?

- Erläutern Sie, warum die Vergübarkeit **Availability** nach dem *ECM Modell* nicht gleich der Vergübarkeit im zuvor behandelten *Reservoir-Modell* ist.
</div>


<div class="alert alert-block alert-warning">

**Solution Aufgabe 06**

1. **Battery Thermal Efficiency**  
   Beschreibt, welcher Anteil der durchgesetzten Energie **nicht** als Wärmeverlust im Innenwiderstand umgesetzt wird.

2. **Battery Round Trip Efficiency (RTE)**  
   Beschreibt, welcher Anteil der zuvor gespeicherten Energie beim Entladen wieder nutzbar abgegeben werden kann.


## 4. Zweiter Zelltyp: `LIB_type2`

Nun wird ein weiterer generischer Lithium-Ionen-Zelltyp `LIB_type2` eingeführt. Auch hier werden Einheiten und normierte Größen konsistent in der Nomenklatur geführt.

In [28]:
def lib2_open_circuit_voltage(soc_pu):
    a1 = -116.2064
    a2 = -22.4512
    a3 = 358.9072
    a4 = 499.9994
    b1 = -0.1572
    b2 = -0.0944
    k0 = 2.0020
    k1 = -3.3160
    k2 = 4.9996
    k3 = -0.4574
    k4 = -1.3646
    k5 = 0.1251

    return (
        k0
        + k1 / (1 + np.exp(a1 * (soc_pu - b1)))
        + k2 / (1 + np.exp(a2 * (soc_pu - b2)))
        + k3 / (1 + np.exp(a3 * (soc_pu - 1)))
        + k4 / (1 + np.exp(a4 * soc_pu))
        + k5 * soc_pu
    )

In [29]:
soc_grid_pu = np.linspace(start=0.0, stop=1.0, num=1000)
ocv_curve_v_v = lib2_open_circuit_voltage(soc_grid_pu)
px.line(
    x=soc_grid_pu,
    y=ocv_curve_v_v,
    template=template,
    labels={"x": "SOC [p.u.]", "y": "OCV [V]"},
    title="Open Circuit Voltage Curve for LIB Type 2",
)

In [30]:
# Extended parameters for LIB2
lib2_cell_nominal_capacity_ah_ah = 200  # Ah
lib2_cell_resistance_ohm = 0.009  # Ohm
lib2_cell_max_charge_c_rate = 1.0  # 1.0 C
lib2_cell_max_discharge_c_rate = 6.6  # 6.6 C
lib2_cell_nominal_voltage_v_v = 3.25  # V
lib2_cell_max_charge_current_a = lib2_cell_nominal_capacity_ah_ah * lib2_cell_max_charge_c_rate
lib2_cell_max_discharge_current_a = lib2_cell_nominal_capacity_ah_ah * lib2_cell_max_discharge_c_rate
lib2_cell_nominal_energy_wh = lib2_cell_nominal_capacity_ah_ah * lib2_cell_nominal_voltage_v_v  # Wh


<div class="alert alert-block alert-info">

**AUFGABE 7**

- Welche Kathodenchemie ist für diese Lithium-Ionen-Zelle plausibel?
- Wiederhole das oben beschriebene Vorgehen, um eine ECM-Simulation für `LIB_type2` zu erstellen.
- Vergleiche anschließend `LIB_type1` und `LIB_type2`:
  - Welche Unterschiede zeigen sich bei den Kennzahlen zur Verfügabrkeit?
  - Für welchen Zelltyp ist eine SoC-Schätzung aus Anwendungssicht voraussichtlich robuster?

Begründe deine Antwort in 2–3 aussagekräftigen Sätzen.

</div>


In [31]:
## Solution - ECM simulation for LIB2

df_lib2_ecm_result = simulate_ecm(
    power_profile_w=power_profile_w,
    cell_nominal_capacity_ah=lib2_cell_nominal_capacity_ah_ah,
    cell_max_charge_current_a=lib2_cell_max_charge_current_a,
    cell_max_discharge_current_a=lib2_cell_max_discharge_current_a,
    initial_soc_pu=0.5,
    cell_resistance_ohm=lib2_cell_resistance_ohm,
    time_step_h=time_step_h,
)

In [32]:
## Solution - ECM plot for LIB2

plot_ecm(
    # rufe hier die Simulation der LIB2 Zelle auf
    df_lib2_ecm_result,
    title="ECM Simulation for LIB2",
)

Berechnen Sie nun die Verfügbarkeit der Battreie Batterie `lib2` mit dem ECM Modell

In [33]:
## solution 0xx - availability for LIB2

availability_pct = calculate_availability(df_lib2_ecm_result)
print("Availability (%) LIB2 ECM:", availability_pct)

Availability (%) LIB2 ECM: 57.65634649065099


In [34]:
lib2_battery_thermal_efficiency_pct, lib2_relative_battery_loss_pct, lib2_battery_round_trip_efficiency_pct = calculate_battery_efficiency(
    df_lib2_ecm_result,
    lib2_cell_resistance_ohm,
)
print("Battery Thermal Efficiency (%):", lib2_battery_thermal_efficiency_pct)
print("Battery Round Trip Efficiency (%):", lib2_battery_round_trip_efficiency_pct)

Battery Thermal Efficiency (%): 96.82470168088552
Battery Round Trip Efficiency (%): 93.45507261978594


<div class="alert alert-block alert-warning">

SOLUTION Aufgabe 07

- Es handelt sich um LFP
- Die Verfügbarkeit von Zelle LIB2 ist im ECM Modell höher als die von Zelle LIB1
- Die Round Trip Effizienz von Zelle LIB1 ist im ECM Modell höher als die von Zelle LIB2

...

#### 2.2.5. Entnehmbare Energie in Abhängigkeit von der C-Rate

Im Folgenden definieren wir Funktionen zur Konstantstormentladung und analysieren deise für unterschiedliche Stromrtaten


<div class="alert alert-block alert-info">

**AUFGABE 07**

- Führen Sie den folgenden Code aus und interpretiere die resultierenden Kurven mit Begriffen aus der Vorlesung.
- Nehmen Sie als Referenz eine "1C" Entladung an - Wie lange kann bei doppelter C-Rate Leistung entnommen werden, bis die Batterie leer ist? Geben Sie einen prozentualen Vergleichswert an und diskutieren Sie diesen.
- Wie unterscheidet sich das Ergebnis vom zuvor verwendeten Reservoirmodell?

</div>


In [35]:
def simulate_constant_power_discharge(c_rate, cell_nominal_capacity_ah=120.0, time_step_h=1 / 60, cell_resistance_ohm= lib1_cell_resistance_ohm):
    soc_pu = 1.0
    power_target_w = -(cell_nominal_capacity_ah * c_rate)
    cell_max_current_a = cell_nominal_capacity_ah * c_rate

    discharge_results = []
    time_h = 0.0

    while soc_pu > 0:
        step_result = simulate_ecm_step(
            soc_pu=soc_pu,
            power_target_w=power_target_w,
            cell_nominal_capacity_ah=cell_nominal_capacity_ah,
            cell_max_charge_current_a=cell_max_current_a,
            cell_max_discharge_current_a=cell_max_current_a,
            time_step_h=time_step_h,
            cell_resistance_ohm=cell_resistance_ohm,
        )
        soc_pu = step_result["soc_pu"]
        time_h += time_step_h
        discharge_results.append((time_h, step_result))

    return pd.DataFrame(
        data=[entry[1] for entry in discharge_results],
        index=[entry[0] for entry in discharge_results],
    )

In [36]:
c_rates = [1 / 60, 1 / 8, 1 / 4, 1 / 2, 1, 2]
series = {c_rate: simulate_constant_power_discharge(c_rate, cell_resistance_ohm=lib1_cell_resistance_ohm) for c_rate in c_rates}

energy_1c_wh = -series[1]["cell_power_w"].sum() * (1 / 60)
print("Total energy discharged (normalized to 1C):")
for c_rate, df_result_c_rate in series.items():
    normalized_energy = -df_result_c_rate["cell_power_w"].sum() * (1 / 60) / energy_1c_wh
    print(f"{c_rate:.2f}C: {normalized_energy:.4f}")

dischargeable_energy_norm = [
    (-df_result_c_rate["cell_power_w"].sum() * (1 / 60)) / energy_1c_wh
    for df_result_c_rate in series.values()
]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=c_rates,
        y=dischargeable_energy_norm,
        mode="lines+markers",
        name="Dischargeable energy",
    )
)

fig.update_layout(
    title="Dischargeable Energy vs C-rate",
    xaxis_title="C-rate",
    yaxis_title="Dischargeable energy (normalized to 1C)",
    template="plotly_white",
)

fig.show()
fig.update_xaxes(type="log")

Total energy discharged (normalized to 1C):
0.02C: 1.0253
0.12C: 1.0226
0.25C: 1.0194
0.50C: 1.0130
1.00C: 1.0000
2.00C: 0.9728


<div class="alert alert-block alert-warning">

### Solution -  Aufgabe 07

- **Peukert-artiger Effekt:** Mit steigender C-Rate sinkt die effektiv nutzbare Energie.
- Ursache sind vor allem **Spannungseinbruch**, **Innenwiderstand** und die frühere Erreichung von Spannungsgrenzen.
- Im einfachen Reservoirmodell tritt dieser Effekt ohne zusätzliche Randbedingungen nicht auf.


## 3. Von der Zelle zum System: einfache Topologie

Nach der Modellierung auf Zellebene folgt die Übertragung auf die Systemebene. Für eine erste Abschätzung genügt ein vereinfachtes, nominales Vorgehen:

1. **Serienschaltung** aus der geforderten Systemspannung bestimmen.
2. **Parallelschaltung** aus der geforderten Systemenergie bestimmen.

Spannungsgrenzen, OCV-Verläufe und Wechselrichteranforderungen werden hier noch nicht vertieft betrachtet.



<div class="alert alert-block alert-info">

**AUFGABE 08**

- Leite eine einfache Funktion her, mit der für einen Anwendungsfall eine geeignete Speicher-Topologie aus **Serien-** und **Parallelschaltung** bestimmt werden kann.
- Für `derive_system_topology(...)` sollen die notwendigen Zellparameter als Funktionsargumente übergeben werden: `required_system_energy_wh`,
    `required_system_voltage_v`, `cell_nominal_voltage_v`, `cell_nominal_energy_wh`
- Die Funktion soll für die vorgegebene Zelle die minimale ganzzahlige Anzahl serieller und paralleler Zellen bzw. Strings liefern, die die Systemanforderungen erfüllen.
- Die Funktion soll eine Topologie berechnen, die möglichst effizient ist (Minimierung der Verluste bei Strombelastung)
- Modulspannungsgrenzen sollen beachtet werden, wie in der Vorlesung erarbeitet

</div>



In [37]:
## Solution Aufgabe 08


def derive_system_topology(
    required_system_energy_wh,
    maximum_system_voltage_v,
    maximum_module_voltage_v,
    cell_min_voltage_v,
    cell_max_voltage_v,
    cell_nominal_energy_wh,
):
    """
    Derive a simple nominal system topology.

    The number of series-connected cells is limited so that neither the module
    voltage limit nor the system voltage limit is exceeded.
    The number of parallel strings is then determined from the energy requirement.
    """
    max_series_cells_by_system = math.floor(maximum_system_voltage_v / cell_max_voltage_v)
    max_series_cells_by_module = math.floor(maximum_module_voltage_v / cell_max_voltage_v)

    num_series_cells = min(max_series_cells_by_system, max_series_cells_by_module)

    if num_series_cells < 1:
        raise ValueError(
            "No valid series cell count possible: voltage limits are too low for the given cell maximum voltage."
        )

    energy_per_series_string_wh = num_series_cells * cell_nominal_energy_wh
    num_parallel_strings = math.ceil(required_system_energy_wh / energy_per_series_string_wh)

    return {
        "num_series_cells": num_series_cells,
        "num_parallel_strings": num_parallel_strings,
        "system_minimum_voltage_v": num_series_cells * cell_min_voltage_v,
        "system_maximum_voltage_v": num_series_cells * cell_max_voltage_v,
        "system_nominal_energy_wh": num_parallel_strings * energy_per_series_string_wh,
        "maximum_module_voltage_v": maximum_module_voltage_v,
        "maximum_system_voltage_v": maximum_system_voltage_v,
    }


<div class="alert alert-block alert-info">

**AUFGABE 09**

Testen Sie die Funktion mit den Beispielwerten aus der Vorlesung 


Bestimme eine geeignete Topologie (**seriell** und **parallel**) für ein Speichersystem mit

- `10 kWh` nutzbarer Nennenergie und
- `800 V` maximaler Betriebsspannung

unter Verwendung von Zellen des Typs `LIB1` und `LIB2`

</div>


In [38]:
## Solution Aufgabe 9

system_system_topology = derive_system_topology(
    required_system_energy_wh=10*1000, # 10 kWh
    maximum_system_voltage_v=800,
    maximum_module_voltage_v=60,    
    cell_min_voltage_v=lib1_open_circuit_voltage(0),  # minimum OCV at 0% SOC
    cell_max_voltage_v=lib1_open_circuit_voltage(1),  # maximum OCV at 100% SOC
    cell_nominal_energy_wh=lib1_cell_nominal_energy_wh)


print("Number of series-connected cells:", system_system_topology["num_series_cells"])
print("Number of parallel strings:", system_system_topology["num_parallel_strings"])
print("Resulting minimum system voltage (V):", system_system_topology["system_minimum_voltage_v"])
print("Resulting maximum system voltage (V):", system_system_topology["system_maximum_voltage_v"])
print("Resulting nominal system energy (Wh):", system_system_topology["system_nominal_energy_wh"])

Number of series-connected cells: 14
Number of parallel strings: 2
Resulting minimum system voltage (V): 45.33410856742224
Resulting maximum system voltage (V): 57.78429849957911
Resulting nominal system energy (Wh): 12432.0


In [39]:
system_system_topology = derive_system_topology(
    required_system_energy_wh=10*1000, # 10 kWh
    maximum_system_voltage_v=800,
    maximum_module_voltage_v=60,    
    cell_min_voltage_v=lib2_open_circuit_voltage(0),  # minimum OCV at 0% SOC
    cell_max_voltage_v=lib2_open_circuit_voltage(1),  # maximum OCV at 100% SOC
    cell_nominal_energy_wh=lib2_cell_nominal_energy_wh)


print("Number of series-connected cells:", system_system_topology["num_series_cells"])
print("Number of parallel strings:", system_system_topology["num_parallel_strings"])
print("Resulting minimum system voltage (V):", system_system_topology["system_minimum_voltage_v"])
print("Resulting maximum system voltage (V):", system_system_topology["system_maximum_voltage_v"])
print("Resulting nominal system energy (Wh):", system_system_topology["system_nominal_energy_wh"])

Number of series-connected cells: 16
Number of parallel strings: 1
Resulting minimum system voltage (V): 32.15699589280152
Resulting maximum system voltage (V): 57.3119999982932
Resulting nominal system energy (Wh): 10400.0



<div class="alert alert-block alert-info">

**Abgabe Ihrer Bearbeitung**

Nach Abschluss der Bearbeitung:

- alle nicht benötigten Plots und Ausgaben löschen,
- das Notebook auf Lesbarkeit und konsistente Benennungen prüfen,
- die Ergebnisse in einer sauberen Abgabefassung speichern.

</div>
